# 📊 Notebook 04 — Model Comparison + SHAP Explainability

**Goal:** Compare all trained models side-by-side and explain predictions using SHAP.

> **Run time:** ~8 min

In [ ]:
# Import tracking, plotting, and evaluation libraries for head-to-head model comparison
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay)

# Rebuild features (same as notebooks 02 & 03)
# Reload the feature store and encode categorical predictors for a consistent comparison dataset
df = spark.table('silver_credit_risk_features').toPandas()
cat_cols = ['AccountType','Branch','Status','LoanPurpose','EmploymentType','HomeOwnership']
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# Define the shared feature list used to compare all candidate credit risk models
feature_cols = [
    'CreditScore','AnnualIncome','LoanAmount','EmploymentYears',
    'DebtToIncomeRatio','NumOpenAccounts','NumDelinquencies',
    'MonthsSinceLastDelinquency','LoanTermMonths',
    'TotalTransactions','TotalLoanAmount','AvgTransactionAmount',
    'NumLoans','MaxTransactionAmount',
    'AccountType_enc','LoanPurpose_enc','EmploymentType_enc','HomeOwnership_enc'
]
# Build the model matrix and binary default label for comparison
X = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y = pd.to_numeric(df['IsDefault'], errors='coerce').fillna(0).astype(int)
# Split the comparison dataset into stratified training and test partitions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


## Step 1 — Load All Models from MLflow

In [ ]:
# Pull all MLflow runs so their logged metrics can be reviewed in one table
runs = mlflow.search_runs(
    experiment_names=['CreditRiskScoring'],
    order_by=['metrics.auc_roc DESC']
)
# Display each tracked run with its AUC-ROC and average precision metrics
print('All model runs:')
print(runs[['tags.mlflow.runName','metrics.auc_roc','metrics.avg_precision']].to_string())


## Step 2 — ROC Curve Comparison

In [ ]:
# Re-train all models quickly for comparison
# Import the model classes that will be retrained for an apples-to-apples comparison
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

# Define the Logistic Regression, Random Forest, and LightGBM candidate models
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()),
                                      ('model',  LogisticRegression(max_iter=1000, C=0.1, random_state=42))]),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'LightGBM':            lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
}

# Train each candidate model, score the test set, and plot its ROC curve
plt.figure(figsize=(10, 6))
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc    = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    results[name] = {'auc': auc, 'model': model, 'y_prob': y_prob}
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

# Add a random baseline and format the ROC comparison chart
plt.plot([0,1],[0,1],'k--', label='Random Baseline')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison — Credit Risk Models')
plt.legend(); plt.tight_layout()
plt.savefig('/tmp/roc_comparison.png', dpi=150)
plt.show()
# Print the final AUC ranking so the champion model is easy to identify
print('\nAUC-ROC Summary:')
for name, r in sorted(results.items(), key=lambda x: -x[1]['auc']):
    print(f'  {name:<25} AUC: {r["auc"]:.4f}')


## Step 3 — Confusion Matrix (Best Model)

In [ ]:
# Select the champion model with the highest AUC-ROC score
best_name  = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
y_pred     = best_model.predict(X_test)

# Build and plot a confusion matrix for the best model on the held-out test set
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Default','Default'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.tight_layout(); plt.show()

# Report business-friendly counts for true approvals, false approvals, and missed defaults
tn, fp, fn, tp = cm.ravel()
print(f'\nBest model: {best_name}')
print(f'True Positives  (Fraud caught):      {tp}')
print(f'False Negatives (Fraud missed):      {fn}')
print(f'False Positives (Good loans blocked): {fp}')
print(f'True Negatives  (Good loans approved): {tn}')


## Step 4 — SHAP Explainability (Best Model)

In [ ]:
# Generate SHAP explanations for the LightGBM champion if the shap package is available
try:
    import shap
    # SHAP TreeExplainer works best with tree-based models
    # Create a TreeExplainer so SHAP values can quantify feature contributions
    explainer = shap.TreeExplainer(results['LightGBM']['model'])
    shap_values = explainer.shap_values(X_test)

    # Summary plot — global feature importance
    # Plot global SHAP importance to highlight the main drivers of default risk
    plt.figure()
    shap.summary_plot(shap_values[1], X_test, feature_names=feature_cols,
                      plot_type='bar', show=False, max_display=12)
    plt.title('SHAP Feature Importance — LightGBM')
    plt.tight_layout(); plt.show()

    # Explain a single high-risk prediction
    # Inspect the highest-risk account to explain one individual prediction
    high_risk_idx = results['LightGBM']['y_prob'].argmax()
    print(f'\nExplaining highest-risk prediction (index {high_risk_idx}):')
    print(f'Default Probability: {results["LightGBM"]["y_prob"][high_risk_idx]:.1%}')
    shap.waterfall_plot(shap.Explanation(
        values=shap_values[1][high_risk_idx],
        base_values=explainer.expected_value[1],
        data=X_test.iloc[high_risk_idx],
        feature_names=feature_cols
    ))
except ImportError:
    # Fall back to the earlier feature-importance view when SHAP is unavailable
    print('Install shap: pip install shap')
    print('Skipping SHAP plots — feature importance from LightGBM shown in Notebook 03.')
